# Pokemon dataset
This dataset contains information on all 802 Pokemon from all Seven Generations of Pokemon. The information contained in this dataset include Base Stats, Performance against Other Types, Height, Weight, Classification, Egg Steps, Experience Points, Abilities, etc. The information was scraped from http://serebii.net/
<hr />

In [50]:
import pandas as pd

master = pd.read_csv('./data/pokemon.csv')
master.drop(inplace=True, columns=['japanese_name','name'])

In [51]:
master.info()  

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 39 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   abilities          801 non-null    str    
 1   against_bug        801 non-null    float64
 2   against_dark       801 non-null    float64
 3   against_dragon     801 non-null    float64
 4   against_electric   801 non-null    float64
 5   against_fairy      801 non-null    float64
 6   against_fight      801 non-null    float64
 7   against_fire       801 non-null    float64
 8   against_flying     801 non-null    float64
 9   against_ghost      801 non-null    float64
 10  against_grass      801 non-null    float64
 11  against_ground     801 non-null    float64
 12  against_ice        801 non-null    float64
 13  against_normal     801 non-null    float64
 14  against_poison     801 non-null    float64
 15  against_psychic    801 non-null    float64
 16  against_rock       801 non-null    fl

In [52]:
missing = {
    'Numero de faltantes: ': master.isna().sum(),
    'Porcentaje de faltantes: ': master.isna().mean()
}
print('='*40)
print(missing)
print('='*40)

{'Numero de faltantes: ': abilities              0
against_bug            0
against_dark           0
against_dragon         0
against_electric       0
against_fairy          0
against_fight          0
against_fire           0
against_flying         0
against_ghost          0
against_grass          0
against_ground         0
against_ice            0
against_normal         0
against_poison         0
against_psychic        0
against_rock           0
against_steel          0
against_water          0
attack                 0
base_egg_steps         0
base_happiness         0
base_total             0
capture_rate           0
classfication          0
defense                0
experience_growth      0
height_m              20
hp                     0
percentage_male       98
pokedex_number         0
sp_attack              0
sp_defense             0
speed                  0
type1                  0
type2                384
weight_kg             20
generation             0
is_legendary           0

In [53]:
master.head()

,abilities,against_bug,against_dark,against_dragon,against_electric,against_fairy,against_fight,against_fire,against_flying,against_ghost,...,percentage_male,pokedex_number,sp_attack,sp_defense,speed,type1,type2,weight_kg,generation,is_legendary
0,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,1,65,65,45,grass,poison,6.9,1,0
1,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,2,80,80,60,grass,poison,13.0,1,0
2,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,3,122,120,80,grass,poison,100.0,1,0
3,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,88.1,4,60,50,65,fire,NaN,8.5,1,0
4,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,88.1,5,80,65,80,fire,NaN,19.0,1,0


In [54]:
master[['percentage_male','type2','height_m']].describe()

,percentage_male,height_m
count,703.000000,781.000000
mean,55.155761,1.163892
std,20.261623,1.080326
min,0.000000,0.100000
25%,50.000000,0.600000
50%,50.000000,1.000000
75%,50.000000,1.500000
max,100.000000,14.500000


In [55]:
master['type2'].nunique()

18

Debido a la presencia de algunos valores nulos, se opto por hacer ciertas imputaciones, entre ellas se hicieron:
1. **percentage_male** -> Debido a la alra desviacion estandar y la similitud entre los 3 cuartiles y que la media es parecida a la mediana, se toma la mediana para imputar.
2. **height_m** -> Aqui se observa una baja desviacion estandar, por lo que se toma la media para imputar
3. **type2** -> Solamente se cambiara el valor 'NaN' por un No, haciendo referencia a que notiene segundo tipo el pokemon

In [56]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import numpy as np


# 1. Separar caracteristicas (X) y variable objetivo (y)
X = master.drop(columns=['is_legendary'])
y = master['is_legendary']

# 2. Primero hacemos el split 
x_train_pre, x_test_pre, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=777, stratify=y, shuffle=True
)

# 3. Crear un Pipeline  para 'type2' (Imputar + Codificar)
type2_pipeline = Pipeline([
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='No')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

# 4. ColumnTransformer definitivo
preprocessor = ColumnTransformer(
    transformers=[
        ('prc_male', SimpleImputer(missing_values=np.nan, strategy='median'), ['percentage_male']),
        ('height', SimpleImputer(missing_values=np.nan, strategy='mean'), ['height_m']),
        ('type2_all', type2_pipeline, ['type2']),
        ('abilities', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ['abilities']),
        ('capture_rate', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ['capture_rate']),
        ('classification', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ['classfication'])
    ],
    remainder='drop'
)

# 5. Ajustar y transformar los sets de datos correctamente
x_train = preprocessor.fit_transform(x_train_pre)
x_test = preprocessor.transform(x_test_pre)


C:\Users\ozzyr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\ozzyr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\ozzyr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg,

In [57]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 38 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   abilities          801 non-null    str    
 1   against_bug        801 non-null    float64
 2   against_dark       801 non-null    float64
 3   against_dragon     801 non-null    float64
 4   against_electric   801 non-null    float64
 5   against_fairy      801 non-null    float64
 6   against_fight      801 non-null    float64
 7   against_fire       801 non-null    float64
 8   against_flying     801 non-null    float64
 9   against_ghost      801 non-null    float64
 10  against_grass      801 non-null    float64
 11  against_ground     801 non-null    float64
 12  against_ice        801 non-null    float64
 13  against_normal     801 non-null    float64
 14  against_poison     801 non-null    float64
 15  against_psychic    801 non-null    float64
 16  against_rock       801 non-null    fl

### KNN

In [58]:
from sklearn.neighbors import KNeighborsClassifier


## Creacion del modelo
knn = KNeighborsClassifier(n_neighbors=8, n_jobs=-1)

## Entrenamiento y predicciones del modelo
knn.fit(x_train, y_train)
y_pred_knn = knn.predict(x_test)

## Cálculo de las métricas
acc_knn = accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn)
rec_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)
cm_knn= confusion_matrix(y_test, y_pred_knn)

print("=" * 50)
print(" " * 10 + "REPORTE DE MÉTRICAS (KNN)")
print("=" * 50)
print(f"Accuracy        : {acc_knn:.4f}")
print(f"Precision       : {prec_knn:.4f}")
print(f"Recall          : {rec_knn:.4f}")
print(f"Puntuacion F1   : {f1_knn:.4f}")
print("-" * 50)
print("Matriz de Confusion:")
print(cm_knn)
print("=" * 50)

          REPORTE DE MÉTRICAS (KNN)
Accuracy        : 0.9565
Precision       : 0.7692
Recall          : 0.7143
Puntuacion F1   : 0.7407
--------------------------------------------------
Matriz de Confusion:
[[144   3]
 [  4  10]]


### SVM


In [59]:
from sklearn.svm import SVC, LinearSVC, NuSVC


# ==========================================
# 1. Support Vector Classification (SVC)
# ==========================================
svc = SVC() 
svc.fit(x_train, y_train)
y_pred_svc = svc.predict(x_test)

acc_svc = accuracy_score(y_test, y_pred_svc)
prec_svc = precision_score(y_test, y_pred_svc)
rec_svc = recall_score(y_test, y_pred_svc)
f1_svc = f1_score(y_test, y_pred_svc)
cm_svc = confusion_matrix(y_test, y_pred_svc)

print("=" * 50)
print(" " * 10 + "REPORTE DE MÉTRICAS (SVC)")
print("=" * 50)
print(f"Accuracy        : {acc_svc:.4f}")
print(f"Precision       : {prec_svc:.4f}")
print(f"Recall          : {rec_svc:.4f}")
print(f"Puntuacion F1   : {f1_svc:.4f}")
print("-" * 50)
print("Matriz de Confusion:")
print(cm_svc)
print("=" * 50)
print("\n")


# =========================================
# 2. Linear Support Vector Classification (LinearSVC)
# ==========================================
linear_svc = LinearSVC(dual=False) 
linear_svc.fit(x_train, y_train)
y_pred_linear_svc = linear_svc.predict(x_test)

acc_linear_svc = accuracy_score(y_test, y_pred_linear_svc)
prec_linear_svc = precision_score(y_test, y_pred_linear_svc)
rec_linear_svc = recall_score(y_test, y_pred_linear_svc)
f1_linear_svc = f1_score(y_test, y_pred_linear_svc)
cm_linear_svc = confusion_matrix(y_test, y_pred_linear_svc)

print("=" * 50)
print(" " * 10 + "REPORTE DE MÉTRICAS (LinearSVC)")
print("=" * 50)
print(f"Accuracy        : {acc_linear_svc:.4f}")
print(f"Precision       : {prec_linear_svc:.4f}")
print(f"Recall          : {rec_linear_svc:.4f}")
print(f"Puntuacion F1   : {f1_linear_svc:.4f}")
print("-" * 50)
print("Matriz de Confusion:")
print(cm_linear_svc)
print("=" * 50)
print("\n")


# ==========================================
# 3. Nu-Support Vector Classification (NuSVC)
# ==========================================
nu_svc = NuSVC(nu=0.05) 
nu_svc.fit(x_train, y_train)
y_pred_nu_svc = nu_svc.predict(x_test)

acc_nu_svc = accuracy_score(y_test, y_pred_nu_svc)
prec_nu_svc = precision_score(y_test, y_pred_nu_svc)
rec_nu_svc = recall_score(y_test, y_pred_nu_svc)
f1_nu_svc = f1_score(y_test, y_pred_nu_svc)
cm_nu_svc = confusion_matrix(y_test, y_pred_nu_svc)

print("=" * 50)
print(" " * 10 + "REPORTE DE MÉTRICAS (NuSVC)")
print("=" * 50)
print(f"Accuracy        : {acc_nu_svc:.4f}")
print(f"Precision       : {prec_nu_svc:.4f}")
print(f"Recall          : {rec_nu_svc:.4f}")
print(f"Puntuacion F1   : {f1_nu_svc:.4f}")
print("-" * 50)
print("Matriz de Confusion:")
print(cm_nu_svc)
print("=" * 50)


          REPORTE DE MÉTRICAS (SVC)
Accuracy        : 0.9130
Precision       : 0.0000
Recall          : 0.0000
Puntuacion F1   : 0.0000
--------------------------------------------------
Matriz de Confusion:
[[147   0]
 [ 14   0]]


          REPORTE DE MÉTRICAS (LinearSVC)
Accuracy        : 0.9752
Precision       : 0.9167
Recall          : 0.7857
Puntuacion F1   : 0.8462
--------------------------------------------------
Matriz de Confusion:
[[146   1]
 [  3  11]]


          REPORTE DE MÉTRICAS (NuSVC)
Accuracy        : 0.9627
Precision       : 0.7857
Recall          : 0.7857
Puntuacion F1   : 0.7857
--------------------------------------------------
Matriz de Confusion:
[[144   3]
 [  3  11]]


C:\Users\ozzyr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Decision Tree Classifier

In [60]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=777) 


dt.fit(x_train, y_train)
y_pred_dt = dt.predict(x_test)

acc_dt = accuracy_score(y_test, y_pred_dt)
prec_dt = precision_score(y_test, y_pred_dt)
rec_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
cm_dt = confusion_matrix(y_test, y_pred_dt)

print("=" * 50)
print(" " * 5 + "REPORTE DE MÉTRICAS (Decision Tree)")
print("=" * 50)
print(f"Accuracy        : {acc_dt:.4f}")
print(f"Precision       : {prec_dt:.4f}")
print(f"Recall          : {rec_dt:.4f}")
print(f"Puntuacion F1   : {f1_dt:.4f}")
print("-" * 50)
print("Matriz de Confusion:")
print(cm_dt)
print("=" * 50)
print("\n")

     REPORTE DE MÉTRICAS (Decision Tree)
Accuracy        : 0.9752
Precision       : 0.9167
Recall          : 0.7857
Puntuacion F1   : 0.8462
--------------------------------------------------
Matriz de Confusion:
[[146   1]
 [  3  11]]




## Modelos de boosting y random forest

In [61]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier

# ==========================================
# 1. AdaBoost Classifier
# ==========================================

## Creación del modelo
ada = AdaBoostClassifier(random_state=42)

## Entrenamiento y predicciones del modelo
ada.fit(x_train, y_train)
y_pred_ada = ada.predict(x_test)

## Cálculo de las métricas
acc_ada = accuracy_score(y_test, y_pred_ada)
prec_ada = precision_score(y_test, y_pred_ada)
rec_ada = recall_score(y_test, y_pred_ada)
f1_ada = f1_score(y_test, y_pred_ada)
cm_ada = confusion_matrix(y_test, y_pred_ada)

## Impresiones para AdaBoost
print("=" * 50)
print(" " * 8 + "REPORTE DE MÉTRICAS (AdaBoost)")
print("=" * 50)
print(f"Accuracy        : {acc_ada:.4f}")
print(f"Precision       : {prec_ada:.4f}")
print(f"Recall          : {rec_ada:.4f}")
print(f"Puntuacion F1   : {f1_ada:.4f}")
print("-" * 50)
print("Matriz de Confusion:")
print(cm_ada)
print("=" * 50)
print("\n")


# ==========================================
# 2. Random Forest Classifier
# ==========================================

## Creación del modelo
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

## Entrenamiento y predicciones del modelo
rf.fit(x_train, y_train)
y_pred_rf = rf.predict(x_test)

## Cálculo de las métricas
acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
cm_rf = confusion_matrix(y_test, y_pred_rf)

## Impresiones para Random Forest
print("=" * 50)
print(" " * 6 + "REPORTE DE MÉTRICAS (Random Forest)")
print("=" * 50)
print(f"Accuracy        : {acc_rf:.4f}")
print(f"Precision       : {prec_rf:.4f}")
print(f"Recall          : {rec_rf:.4f}")
print(f"Puntuacion F1   : {f1_rf:.4f}")
print("-" * 50)
print("Matriz de Confusion:")
print(cm_rf)
print("=" * 50)
print("\n")


# ==========================================
# 3. Gradient Boosting Classifier
# ==========================================

## Creación del modelo
gb = GradientBoostingClassifier(random_state=42)

## Entrenamiento y predicciones del modelo
gb.fit(x_train, y_train)
y_pred_gb = gb.predict(x_test)

## Cálculo de las métricas
acc_gb = accuracy_score(y_test, y_pred_gb)
prec_gb = precision_score(y_test, y_pred_gb)
rec_gb = recall_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb)
cm_gb = confusion_matrix(y_test, y_pred_gb)

## Impresiones para Gradient Boosting
print("=" * 50)
print(" " * 5 + "REPORTE DE MÉTRICAS (Gradient Boosting)")
print("=" * 50)
print(f"Accuracy        : {acc_gb:.4f}")
print(f"Precision       : {prec_gb:.4f}")
print(f"Recall          : {rec_gb:.4f}")
print(f"Puntuacion F1   : {f1_gb:.4f}")
print("-" * 50)
print("Matriz de Confusion:")
print(cm_gb)
print("=" * 50)
print("\n")

        REPORTE DE MÉTRICAS (AdaBoost)
Accuracy        : 0.9689
Precision       : 1.0000
Recall          : 0.6429
Puntuacion F1   : 0.7826
--------------------------------------------------
Matriz de Confusion:
[[147   0]
 [  5   9]]


      REPORTE DE MÉTRICAS (Random Forest)
Accuracy        : 0.9689
Precision       : 1.0000
Recall          : 0.6429
Puntuacion F1   : 0.7826
--------------------------------------------------
Matriz de Confusion:
[[147   0]
 [  5   9]]


     REPORTE DE MÉTRICAS (Gradient Boosting)
Accuracy        : 0.9814
Precision       : 1.0000
Recall          : 0.7857
Puntuacion F1   : 0.8800
--------------------------------------------------
Matriz de Confusion:
[[147   0]
 [  3  11]]




### GaussianNB

In [62]:
from sklearn.naive_bayes import GaussianNB


## Creación del modelo
gnb = GaussianNB()

## Entrenamiento y predicciones del modelo
gnb.fit(x_train, y_train)
y_pred_gnb = gnb.predict(x_test)

## Cálculo de las métricas
acc_gnb = accuracy_score(y_test, y_pred_gnb)
prec_gnb = precision_score(y_test, y_pred_gnb)
rec_gnb = recall_score(y_test, y_pred_gnb)
f1_gnb = f1_score(y_test, y_pred_gnb)
cm_gnb = confusion_matrix(y_test, y_pred_gnb)

## Impresiones para GaussianNB
print("=" * 50)
print(" " * 8 + "REPORTE DE MÉTRICAS (GaussianNB)")
print("=" * 50)
print(f"Accuracy        : {acc_gnb:.4f}")
print(f"Precision       : {prec_gnb:.4f}")
print(f"Recall          : {rec_gnb:.4f}")
print(f"Puntuacion F1   : {f1_gnb:.4f}")
print("-" * 50)
print("Matriz de Confusion:")
print(cm_gnb)
print("=" * 50)
print("\n")

        REPORTE DE MÉTRICAS (GaussianNB)
Accuracy        : 0.8820
Precision       : 0.4138
Recall          : 0.8571
Puntuacion F1   : 0.5581
--------------------------------------------------
Matriz de Confusion:
[[130  17]
 [  2  12]]




### MLPClassifier

In [63]:
from sklearn.neural_network import MLPClassifier

## Creación del modelo
mlp = MLPClassifier(hidden_layer_sizes=(150, 50),
        activation='relu',
        solver='adam',
        learning_rate_init=0.003,
        max_iter=2500,
        random_state=777)

## Entrenamiento y predicciones del modelo
mlp.fit(x_train, y_train)
y_pred_mlp = mlp.predict(x_test)

## Cálculo de las métricas
acc_mlp = accuracy_score(y_test, y_pred_mlp)
prec_mlp = precision_score(y_test, y_pred_mlp)
rec_mlp = recall_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp)
cm_mlp = confusion_matrix(y_test, y_pred_mlp)

## Impresiones para MLPClassifier
print("=" * 50)
print(" " * 7 + "REPORTE DE MÉTRICAS (MLPClassifier)")
print("=" * 50)
print(f"Accuracy        : {acc_mlp:.4f}")
print(f"Precision       : {prec_mlp:.4f}")
print(f"Recall          : {rec_mlp:.4f}")
print(f"Puntuacion F1   : {f1_mlp:.4f}")
print("-" * 50)
print("Matriz de Confusion:")
print(cm_mlp)
print("=" * 50)
print("\n")

       REPORTE DE MÉTRICAS (MLPClassifier)
Accuracy        : 0.9565
Precision       : 0.7692
Recall          : 0.7143
Puntuacion F1   : 0.7407
--------------------------------------------------
Matriz de Confusion:
[[144   3]
 [  4  10]]




## Conclusiones

Tras evaluar 10 algoritmos de clasificación diferentes sobre el dataset de Pokémon, se ha determinado que **Gradient Boosting** es el modelo con mejor desempeño general, alcanzando:

### Resultados del Mejor Algoritmo (Gradient Boosting)
- **Accuracy: 98.14%** - El modelo clasifica correctamente el 98.14% de los Pokémon en el conjunto de prueba
- **Precision: 100%** - Cuando el modelo predice que un Pokémon es legendario, acerta el 100% de las veces (sin falsos positivos)
- **Recall: 78.57%** - Identifica el 78.57% de todos los Pokémon legendarios en el conjunto de prueba
- **F1 Score: 0.8800** - Excelente balance entre precisión y recall

### Comparación con Otros Modelos
Los siguientes algoritmos también mostraron un buen desempeño:
1. **LinearSVC y Decision Tree** (Accuracy: 97.52%) - Rendimiento muy cercano al ganador
2. **AdaBoost y Random Forest** (Accuracy: 96.89%) - Modelos de ensemble con precisión perfecta pero menor recall
3. **NuSVC** (Accuracy: 96.27%) - SVM con parámetro nu optimizado

### Recomendaciones
- **Gradient Boosting** es la mejor opción para clasificar Pokémon legendarios, especialmente por su excelente balance entre precisión y recall
- La precisión perfecta es valiosa para evitar falsos positivos (etiquetar erróneamente un Pokémon como legendario)
- Para aplicaciones donde el recall es crítico, se podría considerar **GaussianNB** que alcanza 85.71% de recall, aunque con menor precisión general
- El preprocesamiento realizado (imputación de valores faltantes, codificación One-Hot) fue efectivo para todos los modelos